# 🧘 Yoga Pose Similarity Search
### Transfer Learning + Milvus Vector Database + Flask Web App

**All data, the Milvus database, and embedding checkpoints are stored on Google Drive**  
so that nothing is lost if your Colab session disconnects.

| Step | Cell(s) |
|---|---|
| Mount Drive & set paths | 1 |
| Install packages | 2 |
| Kaggle auth + download dataset (skipped if already on Drive) | 3 |
| Load ResNet50 feature extractor | 4 |
| Embedding helper | 5 |
| Build / resume Milvus DB | 6 |
| Similarity search function | 7 |
| Quick inline test | 8 |
| ngrok token | 9 |
| Write Flask app | 10 |
| Launch web app | 11 |
| Inline uploader (no ngrok) | 12 |
| Inspect DB stats | 13 |

---
## Cell 1 — Mount Google Drive & Configure Paths

Everything persistent (dataset, Milvus DB, checkpoints) lives under `DRIVE_ROOT`.  
You can change `DRIVE_ROOT` to any folder you prefer inside your Drive.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# ── ⚙️  Edit this path if you want a different Drive folder ──────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/YogaPoseSearch'
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_DATA        = os.path.join(DRIVE_ROOT, 'dataset')          # raw images
DRIVE_DB          = os.path.join(DRIVE_ROOT, 'yoga_milvus.db')   # Milvus DB
DRIVE_CHECKPOINT  = os.path.join(DRIVE_ROOT, 'checkpoint.txt')   # processed image log
DRIVE_KAGGLE      = os.path.join(DRIVE_ROOT, 'kaggle.json')      # cached API key

for d in [DRIVE_ROOT, DRIVE_DATA]:
    os.makedirs(d, exist_ok=True)

print('Google Drive mounted ✓')
print(f'Drive root : {DRIVE_ROOT}')
print(f'Dataset    : {DRIVE_DATA}')
print(f'Milvus DB  : {DRIVE_DB}')
print(f'Checkpoint : {DRIVE_CHECKPOINT}')

---
## Cell 2 — Install Dependencies

In [ ]:
!pip install -q milvus-lite pymilvus torch torchvision flask pyngrok pillow kaggle

---
## Cell 3 — Kaggle Auth & Dataset Download

- If `kaggle.json` is already on Drive it is reused automatically — no re-upload needed.
- If the dataset folder already exists on Drive the download is skipped entirely.

In [ ]:
import shutil, glob
from google.colab import files as colab_files

KAGGLE_DIR = '/root/.kaggle'
os.makedirs(KAGGLE_DIR, exist_ok=True)

# ── Step 1: Get kaggle.json (Drive cache → upload fallback) ──────────────────
if os.path.exists(DRIVE_KAGGLE):
    shutil.copy(DRIVE_KAGGLE, f'{KAGGLE_DIR}/kaggle.json')
    print('kaggle.json loaded from Drive ✓')
else:
    print('kaggle.json not found on Drive — please upload it now:')
    uploaded = colab_files.upload()
    with open(f'{KAGGLE_DIR}/kaggle.json', 'wb') as fh:
        fh.write(uploaded['kaggle.json'])
    # Save to Drive for future sessions
    shutil.copy(f'{KAGGLE_DIR}/kaggle.json', DRIVE_KAGGLE)
    print('kaggle.json saved to Drive for future sessions ✓')

os.chmod(f'{KAGGLE_DIR}/kaggle.json', 0o600)

# ── Step 2: Download dataset (skip if already on Drive) ──────────────────────
existing = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
if existing:
    DATASET_DIR = existing[0]
    print(f'Dataset already on Drive — skipping download ✓')
    print(f'Dataset root: {DATASET_DIR}')
else:
    print('Downloading dataset to Drive (this may take a few minutes)...')
    !kaggle datasets download -d shrutisaxena/yoga-pose-image-classification-dataset \
        -p "{DRIVE_DATA}" --unzip
    existing = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
    DATASET_DIR = existing[0] if existing else DRIVE_DATA
    print(f'Download complete ✓')
    print(f'Dataset root: {DATASET_DIR}')

print(f'Classes found: {len(os.listdir(DATASET_DIR))}')

---
## Cell 4 — ⚙️ Configuration

In [ ]:
# All paths point to Drive
DB_PATH    = DRIVE_DB          # Milvus-lite DB on Drive
COLLECTION = 'yoga_embeddings'
EMB_DIM    = 2048              # ResNet50 penultimate layer
IMG_SIZE   = 224
BATCH_SIZE = 64                # images inserted per Milvus batch
TOP_N      = 5                 # default search results

print('Config set ✓')
print(f'  DB_PATH    = {DB_PATH}')
print(f'  COLLECTION = {COLLECTION}')
print(f'  EMB_DIM    = {EMB_DIM}')

---
## Cell 5 — Load Pre-trained ResNet50 Feature Extractor

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load ResNet50 pre-trained on ImageNet, strip the classification head
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('ResNet50 feature extractor ready ✓')

---
## Cell 6 — Embedding Helper

In [ ]:
def get_embedding(image_path: str) -> np.ndarray:
    """
    Returns an L2-normalized 2048-d embedding for a single image
    using the pre-trained ResNet50 feature extractor.
    """
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = feature_extractor(tensor).squeeze().cpu().numpy()
    emb = emb / (np.linalg.norm(emb) + 1e-10)
    return emb.astype(np.float32)

print('get_embedding() defined ✓')

---
## Cell 7 — Build / Resume Milvus Vector Database (Drive-backed)

**Checkpoint logic:**
- The path of every successfully embedded image is written to `checkpoint.txt` on Drive.
- If the cell is interrupted and re-run, already-processed images are skipped automatically.
- The Milvus `.db` file lives on Drive, so all previously inserted vectors are preserved.

Expect ~5–15 min for a full run (faster on GPU).

In [ ]:
from pymilvus import MilvusClient
import time

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# ── Load checkpoint (images already embedded in a previous run) ──────────────
if os.path.exists(DRIVE_CHECKPOINT):
    with open(DRIVE_CHECKPOINT, 'r') as fh:
        already_done = set(line.strip() for line in fh if line.strip())
    print(f'Checkpoint loaded — {len(already_done)} images already processed ✓')
else:
    already_done = set()
    print('No checkpoint found — starting fresh.')

# ── Open Milvus (creates file on Drive if it does not exist) ─────────────────
client = MilvusClient(DB_PATH)

if not client.has_collection(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        dimension=EMB_DIM,
        metric_type='COSINE',
        auto_id=True,
    )
    print(f'Created collection "{COLLECTION}" ✓')
else:
    print(f'Collection "{COLLECTION}" already exists — resuming ✓')

# ── Collect all image paths, skip already-processed ones ────────────────────
all_images = [
    os.path.join(root, fname)
    for root, _, files in os.walk(DATASET_DIR)
    for fname in files
    if os.path.splitext(fname)[1].lower() in VALID_EXTS
]
pending = [p for p in all_images if p not in already_done]
print(f'Total images  : {len(all_images)}')
print(f'Already done  : {len(already_done)}')
print(f'To process    : {len(pending)}')

# ── Embed and insert in batches ──────────────────────────────────────────────
data_buffer   = []
done_buffer   = []
total_new     = 0
skipped       = 0
start         = time.time()

checkpoint_fh = open(DRIVE_CHECKPOINT, 'a')  # append mode — preserves old entries

try:
    for i, fpath in enumerate(pending):
        label = os.path.basename(os.path.dirname(fpath))
        try:
            emb = get_embedding(fpath)
            data_buffer.append({'vector': emb, 'label': label, 'image_path': fpath})
            done_buffer.append(fpath)
            total_new += 1
        except Exception:
            skipped += 1
            continue

        if len(data_buffer) >= BATCH_SIZE:
            client.insert(collection_name=COLLECTION, data=data_buffer)
            # Write this batch to the checkpoint file immediately
            checkpoint_fh.write('\n'.join(done_buffer) + '\n')
            checkpoint_fh.flush()
            data_buffer, done_buffer = [], []

        if (i + 1) % 500 == 0:
            elapsed = time.time() - start
            print(f'  [{i+1}/{len(pending)}] inserted={total_new}  skipped={skipped}  '
                  f'elapsed={elapsed:.0f}s')

    # Insert any remaining
    if data_buffer:
        client.insert(collection_name=COLLECTION, data=data_buffer)
        checkpoint_fh.write('\n'.join(done_buffer) + '\n')
        checkpoint_fh.flush()

finally:
    checkpoint_fh.close()
    client.close()

elapsed = time.time() - start
print(f'\n✅ Done!  New embeddings inserted: {total_new}  |  Skipped: {skipped}  |  Time: {elapsed:.1f}s')
print(f'Milvus DB saved to: {DB_PATH}')
print(f'Checkpoint saved to: {DRIVE_CHECKPOINT}')

---
## Cell 8 — Similarity Search Function

In [ ]:
def find_similar(image_path: str, top_n: int = TOP_N) -> list:
    """
    Query Milvus for the top_n most similar yoga pose images.
    Returns [{score, label, image_path}, ...]
    """
    query_emb = get_embedding(image_path)
    client = MilvusClient(DB_PATH)
    results = client.search(
        collection_name=COLLECTION,
        data=[query_emb],
        limit=top_n,
        output_fields=['label', 'image_path'],
        search_params={'metric_type': 'COSINE'},
    )
    client.close()
    return [
        {'score': round(float(h['distance']), 4),
         'label': h['entity']['label'],
         'image_path': h['entity']['image_path']}
        for h in results[0]
    ]

print('find_similar() defined ✓')

---
## Cell 9 — Quick Inline Test

Picks a random image from the dataset and shows the top 5 most similar poses.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}
all_images = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_DIR)
    for f in files
    if os.path.splitext(f)[1].lower() in VALID_EXTS
]

query_path  = random.choice(all_images)
query_label = os.path.basename(os.path.dirname(query_path))
print(f'Query image : {query_path}  (class: {query_label})')

hits = find_similar(query_path, top_n=5)

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(mpimg.imread(query_path))
axes[0].set_title(f'QUERY\n{query_label}', fontsize=9, color='blue', fontweight='bold')
axes[0].axis('off')

for i, hit in enumerate(hits, start=1):
    try:
        axes[i].imshow(mpimg.imread(hit['image_path']))
    except Exception:
        pass
    axes[i].set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
    axes[i].axis('off')

plt.suptitle('Yoga Pose Similarity Search — Top 5 Results', fontsize=12)
plt.tight_layout()
plt.show()

---
## Cell 10 — ⚙️ ngrok Authentication

Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken  
Only needed if you want the Flask web app. Otherwise skip to **Cell 12**.

In [ ]:
NGROK_TOKEN = ''   # ← paste your ngrok token here

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print('ngrok token set ✓')
else:
    print('⚠️  No ngrok token — skip to Cell 12 for the inline uploader instead.')

---
## Cell 11 — Write Flask App Files

In [ ]:
os.makedirs('/content/flask_app/templates', exist_ok=True)
os.makedirs('/content/flask_app/static/uploads', exist_ok=True)

APP_PY = f'''
import os, shutil, sys
sys.path.insert(0, "/content")
from flask import Flask, request, render_template
from pymilvus import MilvusClient
import torch, torchvision.transforms as T, numpy as np
from torchvision import models
from PIL import Image

DB_PATH    = "{DRIVE_DB}"
COLLECTION = "yoga_embeddings"
IMG_SIZE   = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
fe = torch.nn.Sequential(*list(resnet.children())[:-1])
fe.eval().to(device)

preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def get_embedding(path):
    img = Image.open(path).convert("RGB")
    t = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        e = fe(t).squeeze().cpu().numpy()
    e = e / (np.linalg.norm(e) + 1e-10)
    return e.astype(np.float32)

def find_similar(path, top_n=5):
    q = get_embedding(path)
    c = MilvusClient(DB_PATH)
    res = c.search(
        collection_name=COLLECTION, data=[q], limit=top_n,
        output_fields=["label", "image_path"],
        search_params={{"metric_type": "COSINE"}})
    c.close()
    return [{{"score": round(float(h["distance"]),4),
              "label": h["entity"]["label"],
              "image_path": h["entity"]["image_path"]}} for h in res[0]]

app = Flask(__name__, template_folder="templates", static_folder="static")
UPLOAD = "static/uploads"
os.makedirs(UPLOAD, exist_ok=True)
ALLOWED = {{"jpg", "jpeg", "png"}}

@app.route("/", methods=["GET", "POST"])
def index():
    results, query_image = [], None
    if request.method == "POST":
        f = request.files.get("image")
        if f and f.filename.rsplit(".",1)[-1].lower() in ALLOWED:
            save = os.path.join(UPLOAD, f.filename)
            f.save(save)
            query_image = f.filename
            top_n = int(request.form.get("top_n", 5))
            for hit in find_similar(save, top_n):
                dest_name = hit["image_path"].replace("/", "_")
                dest = os.path.join("static/uploads", dest_name)
                if not os.path.exists(dest):
                    shutil.copy(hit["image_path"], dest)
                results.append({{"score": hit["score"], "label": hit["label"], "filename": dest_name}})
    return render_template("index.html", query_image=query_image, results=results)

if __name__ == "__main__":
    app.run(port=5000)
'''

INDEX_HTML = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width,initial-scale=1"/>
  <title>Yoga Pose Search</title>
  <style>
    body{font-family:Arial,sans-serif;max-width:960px;margin:40px auto;padding:0 20px;background:#f5f5f5}
    h1{color:#4a4a8a}
    form{background:#fff;padding:20px;border-radius:8px;box-shadow:0 2px 6px rgba(0,0,0,.1)}
    button{background:#4a4a8a;color:#fff;padding:10px 24px;border:none;border-radius:5px;cursor:pointer;font-size:1rem}
    button:hover{background:#36367a}
    .grid{display:flex;flex-wrap:wrap;gap:16px;margin-top:16px}
    .card{background:#fff;border-radius:8px;padding:12px;box-shadow:0 2px 6px rgba(0,0,0,.1);text-align:center;width:160px}
    .card img{width:140px;height:140px;object-fit:cover;border-radius:4px}
    .label{font-weight:bold;margin-top:8px;color:#4a4a8a;text-transform:capitalize;font-size:.9rem}
    .score{font-size:.8rem;color:#777}
    .qimg{width:200px;height:200px;object-fit:cover;border-radius:8px;border:3px solid #4a4a8a}
    .section{margin-top:28px}
  </style>
</head>
<body>
<h1>&#129336; Yoga Pose Similarity Search</h1>
<form method="POST" enctype="multipart/form-data">
  <label><strong>Upload a yoga pose image:</strong></label><br/>
  <input type="file" name="image" accept="image/*" required/><br/><br/>
  <label><strong>Results to return:</strong>
    <input type="number" name="top_n" value="5" min="1" max="20" style="width:60px;margin-left:8px"/>
  </label><br/><br/>
  <button type="submit">Find Similar Poses</button>
</form>
{% if query_image %}
<div class="section">
  <h2>Your Upload</h2>
  <img class="qimg" src="/static/uploads/{{ query_image }}" alt="query"/>
</div>
{% endif %}
{% if results %}
<div class="section">
  <h2>Most Similar Poses</h2>
  <div class="grid">
    {% for r in results %}
    <div class="card">
      <img src="/static/uploads/{{ r.filename }}" alt="{{ r.label }}"/>
      <div class="label">{{ r.label }}</div>
      <div class="score">score: {{ r.score }}</div>
    </div>
    {% endfor %}
  </div>
</div>
{% endif %}
</body>
</html>
'''

with open('/content/flask_app/app.py', 'w') as fh:
    fh.write(APP_PY)
with open('/content/flask_app/templates/index.html', 'w') as fh:
    fh.write(INDEX_HTML)

print('Flask app files written ✓')

---
## Cell 12 — Launch Flask App (ngrok public URL)

Click the URL that appears to open the web app. Interrupt the kernel to stop.

In [ ]:
import subprocess, time, threading
from pyngrok import ngrok

os.chdir('/content/flask_app')

def run_flask():
    subprocess.run(['python', 'app.py'])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)

public_url = ngrok.connect(5000)
print(f'\n🌐 Web App URL: {public_url}')
print('Open the URL above in your browser.')
print('Interrupt the kernel (Runtime → Interrupt) to stop the server.')

while True:
    time.sleep(60)

---
## Cell 13 — (Alternative) Inline Upload & Search — No ngrok needed

Upload any image from your computer and see top-5 similar poses plotted inline.

In [ ]:
from google.colab import files as colab_files
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Upload a yoga pose image:')
uploaded = colab_files.upload()

for fname, data in uploaded.items():
    query_path = f'/content/{fname}'
    with open(query_path, 'wb') as fh:
        fh.write(data)

    print(f'\nSearching for poses similar to: {fname}')
    hits = find_similar(query_path, top_n=5)

    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    axes[0].imshow(mpimg.imread(query_path))
    axes[0].set_title('QUERY', fontsize=10, color='blue', fontweight='bold')
    axes[0].axis('off')

    for i, hit in enumerate(hits, start=1):
        try:
            axes[i].imshow(mpimg.imread(hit['image_path']))
        except Exception:
            pass
        axes[i].set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
        axes[i].axis('off')

    plt.suptitle(f'Top 5 Similar Yoga Poses to "{fname}"', fontsize=12)
    plt.tight_layout()
    plt.show()

    print('\nTop results:')
    for h in hits:
        print(f"  [{h['score']:.4f}]  {h['label']:30s}  {h['image_path']}")

---
## Cell 14 — Inspect DB Stats

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient(DB_PATH)
stats = client.get_collection_stats(COLLECTION)
print(f'Collection   : {COLLECTION}')
print(f'Total rows   : {stats["row_count"]}')
print(f'Embedding dim: {EMB_DIM}')
print(f'Metric type  : COSINE')
print(f'DB location  : {DB_PATH}')

# Show Drive folder summary
print('\nDrive folder contents:')
for f in sorted(os.listdir(DRIVE_ROOT)):
    fpath = os.path.join(DRIVE_ROOT, f)
    size_mb = os.path.getsize(fpath) / 1e6 if os.path.isfile(fpath) else 0
    kind = 'file' if os.path.isfile(fpath) else 'dir'
    print(f'  [{kind}] {f}  {size_mb:.1f} MB' if kind == 'file' else f'  [{kind}] {f}/')

client.close()